<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/07-joins-deep-dive/03-ambiguous-columns.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 7.3 — Ambiguous columns after joins

`orders.country` (shipping) and `customers.country` (home) share a
name on purpose. We trigger the ambiguity error, then work through
the four fixes and the self-join trap.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.utils import AnalysisException

spark = (
    SparkSession.builder
    .appName("7.3-ambiguous-columns")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA_DIR}/customers.csv", header=True, inferSchema=True)

## Trigger it

The JOIN succeeds — duplicate names are legal in a DataFrame. Only
RESOLVING the name fails. That's why this bug surfaces far from its cause.

In [ ]:
joined = orders.join(customers,
                     orders["customer_id"] == customers["customer_id"])
print(joined.columns)   # customer_id twice, country twice — and no error yet

In [ ]:
try:
    joined.select("country")
except AnalysisException as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

In [ ]:
# writes reject duplicates too — the crash site can be a nightly parquet job
try:
    joined.limit(1).write.mode("overwrite").parquet("output/boom")
except AnalysisException as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

## Fix 0 — join on the NAME: USING semantics merges the key

One `customer_id` in the output... but `country` is still doubled.
In a full outer join the merged key is coalesce(left, right): c021's
id shows up even though every left column is null.

In [ ]:
using = orders.join(customers, on="customer_id", how="full")
print("customer_id count:", using.columns.count("customer_id"),
      "| country count:", using.columns.count("country"))

using.filter(col("order_id").isNull()).select("customer_id", "name", "segment").show()

## Fix 1 — rename before the join (production default)

The collision was a naming bug all along: these are two different facts.

In [ ]:
orders_r = orders.withColumnRenamed("country", "ship_country")
custs_r = customers.withColumnRenamed("country", "home_country")

enriched = (
    orders_r.join(custs_r, "customer_id", "left")
    .select("order_id", "customer_id", "product", "quantity",
            "unit_price", "ship_country", "home_country", "segment")
)
enriched.show(5)

In [ ]:
# and now a question the collision was hiding: who ships abroad?
enriched.filter(
    col("ship_country").isNotNull() & (col("ship_country") != col("home_country"))
).select("order_id", "customer_id", "ship_country", "home_country").show()

## Fix 2 — alias namespaces, SQL-style qualification

In [ ]:
j = orders.alias("o").join(customers.alias("c"),
                           col("o.customer_id") == col("c.customer_id"))
j.select(
    "o.order_id",
    col("o.country").alias("ship_country"),
    col("c.country").alias("home_country"),
).show(5)

## Fix 3 — drop the loser: string vs column ref

`drop("name")` with a string drops EVERY column of that name.
The source-DataFrame column ref removes exactly one.

In [ ]:
j2 = orders.join(customers, "customer_id", "left")

print("string drop :", j2.drop("country").columns.count("country"))            # 0 — both gone!
print("ref drop    :", j2.drop(customers["country"]).columns.count("country")) # 1 — surgical

## The self-join trap

Same DataFrame on both sides = same internal attribute IDs on both
sides. The condition below can't mean anything, and modern Spark
refuses to guess (spark.sql.analyzer.failAmbiguousSelfJoin).

In [ ]:
try:
    orders.join(orders, orders["customer_id"] == orders["customer_id"]).count()
except AnalysisException as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

In [ ]:
# aliases mint fresh namespaces. Same-customer order pairs, each pair once:
o1, o2 = orders.alias("o1"), orders.alias("o2")
pairs = o1.join(
    o2,
    (col("o1.customer_id") == col("o2.customer_id"))
    & (col("o1.order_id") < col("o2.order_id")),
).select(
    col("o1.customer_id"),
    col("o1.order_id").alias("first_order"),
    col("o2.order_id").alias("later_order"),
)
pairs.orderBy("customer_id", "first_order", "later_order").show(10)

## Exercises

1. Reproduce the derived-self-join variant: join `orders` to
   `orders.groupBy("customer_id").agg(F.count("*").alias("n"))` with an
   EXPRESSION condition on customer_id. Does it error, warn, or silently
   work? Fix it two ways (alias; rename-agg-key).
2. The `pairs` self-join uses `<`. Rerun with `!=` and predict the count
   first. Then compute "days between consecutive orders per customer" —
   you'll want `datediff` — and note this becomes one line with window
   functions (`lag`, Module 8 teaser).
3. Write `same_name_collisions(df1, df2)`: returns the set of non-key
   column names that would collide in a join. Use it to build
   `safe_join(df1, df2, on, how)` that auto-prefixes collisions
   (`left_`/`right_`) before joining — a mini 6.4-style utility.
4. In SQL (temp views): SELECT with `USING(customer_id)` and with
   `ON o.customer_id = c.customer_id`, and compare output schemas —
   confirm the DataFrame `on=` string/expression split mirrors SQL exactly.

In [ ]:
# your exercise code here